In [ ]:
import numpy as np

df[["peak-rpm","highway-mpg","price"]].corr()

# regplot 画的是：散点 + 回归线（看“拟合长什么样”）
# residplot 画的是：每个点到回归线的垂直距离（看“拟合差多少”）
width = 12
height = 10
plt.figure(figsize=(width, height))
# 解析——残差图：检验线性模型是否合适
# 一行代码内部其实做了三件事：
# 对 (highway-mpg, price) 做线性拟合，得到Yhat
# 计算每个样本的残差 Y-Yhat
# 以 X 为横轴、残差为纵轴画散点图（图中横线 y=0就是那条"理想回归线”的位置）
# residual plot：自动完成 拟合→预测→算残差→画图 全流程
#自变量和因变量都是直接传 Series 列，和regplot不同
sns.residplot(x=df['highway-mpg'], y=df['price'])
plt.show()
# 残差图的形态	            结论
# 随机散布在 0 上下，无规律	线性模型合适，误差是随机噪音
# 呈现曲线趋势（U形/弧形）	数据有非线性关系，线性模型不合适
# 本 lab 的实际结果：残差图呈现明显的弯曲模式——低 mpg 区域残差为负（模型高估价格），中间区域为正，高 mpg 区域又变化。这说明价格和 highway-mpg 不是纯线性关系。
# 真实 < 预测  为负，高估价格；真实 > 预测 为正，低估价格
#这张图诊断出了简单线性回归不足，所以后面引入了多项式回归

# regplot	residplot
# 横轴	    highway-mpg	highway-mpg
# 纵轴	    price（原始值）	残差（误差值）
# 回归线	显示出来	隐含在 y=0 参考线
# 回答的问题    拟合长什么样？	拟合得好不好？


#根据拟合好的公式进行预测（多项）
Y_hat = lm.predict(Z)
plt.figure(figsize=(width, height))

# 参数	                           含义
# sns.distplot	            seaborn 的分布图函数，这里只画曲线不画柱子（该函数在新版 seaborn 中已弃用，替代方案见文末）
# df['price']	            数据来源：201 辆车的真实价格列
# hist=False	            不画直方图，只画平滑的概率密度曲线（KDE）
# color="r"	                 曲线颜色为红色（red）
# label="Actual Value"	    图例标签为 "Actual Value"（真实值）
# ax1 = ...	                把返回的坐标轴对象存到 ax1，供下一行在其上叠加第二条曲线
ax1 = sns.distplot(df['price'], hist=False, color="r", label="Actual Value")

# 这行是上一行的“配对”代码——把模型预测价格画成蓝色曲线，叠加到真实价格的图上。
# 参数	含义
# Y_hat	上一节             lm.predict(Z) 的结果：模型对 201 辆车的预测价格数组（多元线性回归，4 个特征）
# hist=False	        不画直方图，只画平滑的核密度估计曲线（KDE）——可以理解为一条“概率密度曲线”，反映各价格附近的车有多密集
# color="b"	            曲线颜色为蓝色（blue），与上一行的红色真实值形成对比
# label="Fitted Values"	图例标签 "Fitted Values"（拟合值/预测值），与上一行的 "Actual Value" 对应
# ax=ax1	            最关键的参数：指定画到 ax1 这个坐标系上
# 这实现了叠加绘图——红蓝两条曲线共享同一坐标系，才能直接对比
sns.distplot(Y_hat, hist=False, color="b", label="Fitted Values" , ax=ax1)


# 语句	画的内容	数据来源	颜色
# 第一句	真实价格的分布曲线	df['price']（数据集里每辆车实际的售价）	🔴 红
# 第二句	预测价格的分布曲线	Y_hat（模型 lm.predict(Z) 算出来的价格）🔵 蓝

#等价写法
# ax1 = sns.kdeplot(df['price'], color="r", label="Actual Value")
# sns.kdeplot(Y_hat, color="b", label="Fitted Values", ax=ax1)

#设置图标题，x轴标签"价格（美元）"，y轴标签"车辆占比"，因为用的是 KDE 密度曲线（hist=False），纵轴表示的不是车的数量，而是该价格附近的车有多密集——曲线越高，说明这个价位的车越常见。
plt.title('Actual vs Fitted Values for Price')
plt.xlabel('Price (in dollars)')
plt.ylabel('Proportion of Cars')

plt.show()
# 关闭当前图形窗口，释放画布。
plt.close()

# 总结
# 图类型	使用的 seaborn 函数	适用模型	出现位置
# 回归图	sns.regplot()	只有简单线性回归（1 个自变量）	评估 highway-mpg 单变量模型
# 残差图	sns.residplot()	只有简单线性回归	检验线性假设是否成立
# 分布图	sns.distplot()	多元线性回归、其他模型	就是这段代码


# 参数	含义	实际调用时传入的值
# model	已拟合好的多项式函数（如 np.poly1d 对象 p）	三次多项式模型
# independent_variable	自变量（原始数据点）	df['highway-mpg']
# dependent_variabble	因变量（原始数据点，注意源码里这个拼写就是错的，只是参数名不影响运行）	df['price']
# Name	x 轴标签文字	'highway-mpg'
def PlotPolly(model, independent_variable, dependent_variabble, Name):
    # 生成 100 个等间距的 x 值，范围从 15 到 55（覆盖 highway-mpg 的取值区间）。作用是生成一条平滑曲线的“骨架”——如果只用原始的 201 个散点连线，曲线会锯齿状不光滑。
    x_new = np.linspace(15, 55, 100)

    # 把多项式模型套用到这 100 个新 x 值上，算出对应的 y 值。p 是 np.poly1d 对象，可以像函数一样直接调用：p(30) 就等于把 mpg=30 代入三次多项式算价格。得到 100 个 (x, y) 点，连起来就是拟合曲线。
    y_new = model(x_new)

    # 这是本函数的核心——一次 plt.plot 同时画两组数据：
    # 数据	样式	含义
    # independent_variable, dependent_variabble	'.'	🔵 散点：201 辆车的真实数据（mpg, price）
    # x_new, y_new	'-'	                            📈 实线：多项式拟合曲线
    # 这样能直观看出曲线“穿过”了多少数据点——贴得越紧，拟合越好。
    plt.plot(independent_variable, dependent_variabble, '.', x_new, y_new, '-')
    plt.title('Polynomial Fit with Matplotlib for Price ~ Length')
    # gca = Get Current Axes，获取当前坐标轴对象，为下一行修改背景色做准备。
    ax = plt.gca()
    # 把绘图区背景设为浅灰色。RGB 三通道都是 0.898（约 229/255），纯装饰，让图更好看。
    ax.set_facecolor((0.898, 0.898, 0.898))
    # gcf = Get Current Figure，获取当前画布对象。这里只是赋值保存、后面没用它，属于可有可无的一行。
    fig = plt.gcf()
    # 设置轴标签：x 轴是传入的变量名（highway-mpg），y 轴是“车价”。
    plt.xlabel(Name)
    plt.ylabel('Price of Cars')

    # 显示图像并关闭画布——和上一节分布图的作用一样。
    plt.show()
    # 关闭画布
    plt.close()


x = df['highway-mpg']
y = df['price']

# 之前简单线性回归拟合这条数据效果不好（R² 仅 0.497，残差图也不随机），所以改用三次多项式：
# np.polyfit(x, y, 3)：用最小二乘法拟合一个 3 次多项式
# 返回值 f 是一个系数数组，从最高次到常数项排列：[b₃, b₂, b₁, a]
# 实际输出类似：array([ 1.8794e-07, -1.0450e-03, 1.4694e+00, 1.4694e+02])
#相当于先把系数算出来
f = np.polyfit(x, y, 3)

# np.poly1d(f)：把系数数组 f 包装成一个多项式对象
# 这个对象可以像函数一样直接调用：
# p(30) → 把 mpg=30 代入多项式，返回预测车价
# 甚至支持向量输入：p(np.array([20, 30, 40])) → 返回 3 个预测价
# 它还能参与运算，比如 p - q（多项式相减）、求导 p.deriv()
# 没有这一步，你手里只有一串系数，还得手写 b3*x**3 + b2*x**2 + b1*x + a；有了它，直接 p(x) 就行。这也是为什么后面的 PlotPolly 里能写 y_new = model(x_new)——model 就是这个 p。
#这就是个公式，能把系数都填进去，形成多项式对象
p = np.poly1d(f)

PlotPolly(p, x, y, 'highway-mpg')
# 把系数数组直接显示出来
np.polyfit(x, y, 3)

# 这个函数plotpolly，其实就是对比真实价格和预测价格的，只不过这次预测价格采用的是三次多项式
#           分布图（MLR 那张）	    PlotPolly（多项式这张）
# 模型	多元线性回归（4 个特征）	三次多项式（1 个特征：highway-mpg）
# 真实值	红色密度曲线 df['price']	蓝色散点 '.'（每个点 = 一辆车的真实 mpg 和真实价格）
# 预测值	蓝色密度曲线 Y_hat	平滑拟合曲线 '-'（p(x_new) 算出的预测价格）
# 呈现方式	看两条分布是否重叠	看曲线穿过多少个散点

# 这两个可视化干的都是同一件事——“真实价格 vs 模型预测价格”的对比评估。区别只在于：
# 多元线性回归 → 只能看分布贴不贴合
# 三次多项式（单变量）→ 可以直接看曲线贴不贴合散点，更直观